In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import dump

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "data").exists() and (cur / "report").exists():
            return cur
        cur = cur.parent
    return start.resolve()


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "report"
FIG_DIR = REPORT_DIR / "figures"
MET_DIR = REPORT_DIR / "metrics"
PREP_DIR = PROJECT_ROOT / "preprocessing"

for d in [MODELS_DIR, FIG_DIR, MET_DIR, PREP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed parts dir:", PROCESSED_DIR)

RANDOM_STATE = 42
CHUNK_SIZE = 50_000


def list_parts(prefix: str) -> list[Path]:
    parts = sorted(PROCESSED_DIR.glob(f"{prefix}_part_*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No Parquet parts found for prefix='{prefix}' in {PROCESSED_DIR}")
    return parts


def detect_label_col(df: pd.DataFrame) -> str:
    for c in df.columns:
        if str(c).strip().lower() == "label":
            return c
    raise KeyError("Label column not found (strip-lower == 'label').")


def coerce_binary_labels(series: pd.Series) -> pd.Series:
    s = series.copy()
    if s.dtype == object:
        ss = s.astype(str).str.strip()
        mapped = ss.map({"BENIGN": 0, "DDoS": 1})
        mapped = mapped.where(~mapped.isna(), (ss != "BENIGN").astype(int))
        return pd.to_numeric(mapped, errors="coerce")
    return pd.to_numeric(s, errors="coerce")


def read_xy_part(p: Path, feature_cols: list[str], label_col: str) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_parquet(p, columns=feature_cols + [label_col])
    y = coerce_binary_labels(df[label_col])
    valid = y.isin([0, 1]).to_numpy()
    if valid.sum() == 0:
        return np.empty((0, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.int64)
    X = df.loc[valid, feature_cols].to_numpy(dtype=np.float32, copy=False)
    yv = y.loc[valid].to_numpy(dtype=np.int64, copy=False)
    return X, yv


def stream_label_counts(parts: list[Path], feature_cols: list[str], label_col: str) -> dict[int, int]:
    counts = {0: 0, 1: 0}
    for p in parts:
        _, y = read_xy_part(p, feature_cols, label_col)
        if y.size == 0:
            continue
        counts[0] += int(np.sum(y == 0))
        counts[1] += int(np.sum(y == 1))
    return counts


def proportions(counts: dict[int, int]) -> dict[int, float]:
    total = sum(counts.values())
    return {k: (counts[k] / total if total else 0.0) for k in counts}


def save_cm(cm: np.ndarray, title: str, out_path: Path) -> None:
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(int(v)), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_roc(y_true: np.ndarray, y_score: np.ndarray, title: str, out_path: Path) -> float:
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"{title} (AUC={auc:.3f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return auc


def eval_stream(
    parts: list[Path],
    feature_cols: list[str],
    label_col: str,
    predict_fn,
    score_fn=None,
    scaler: StandardScaler | None = None,
    need_scores: bool = True,
):
    cm = np.zeros((2, 2), dtype=np.int64)
    y_true_all = []
    y_pred_all = []
    y_score_all = []

    for p in parts:
        X, y = read_xy_part(p, feature_cols, label_col)
        if y.size == 0:
            continue

        if scaler is not None:
            X = scaler.transform(X)

        y_pred = predict_fn(X)
        cm += confusion_matrix(y, y_pred, labels=[0, 1])

        y_true_all.append(y)
        y_pred_all.append(y_pred)

        if need_scores and score_fn is not None:
            y_score_all.append(score_fn(X))

    if not y_true_all:
        raise ValueError("No valid labels found in this split after coercion/filtering.")

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")

    if need_scores and score_fn is not None:
        y_score = np.concatenate(y_score_all)
        auc = roc_auc_score(y_true, y_score)
    else:
        y_score = None
        auc = None

    return cm, acc, f1m, auc, y_true, y_score


main_figs = [
    "fig_class_balance_by_split.png",
    "fig_cm_dummy_val.png",
    "fig_cm_dummy_test.png",
    "fig_cm_sgd_val.png",
    "fig_cm_sgd_test.png",
    "fig_roc_sgd_test.png",
    "fig_coefficients_sgd_logreg.png",
    "fig_cm_rf_val.png",
    "fig_cm_rf_test.png",
    "fig_roc_rf_test.png",
    "fig_feature_importance_rf.png",
]
for fn in main_figs:
    fp = FIG_DIR / fn
    if fp.exists():
        fp.unlink()

for fp in [MET_DIR / "metrics_summary.csv", PREP_DIR / "rf_top_features.csv"]:
    if fp.exists():
        fp.unlink()

for fp in [MODELS_DIR / "sgd_logreg_stream.joblib", MODELS_DIR / "rf_sample.joblib"]:
    if fp.exists():
        fp.unlink()

# Parts + schema
train_parts = list_parts("train_clean")
val_parts = list_parts("val_clean")
test_parts = list_parts("test_clean")

sample_df = pd.read_parquet(train_parts[0])
LABEL_COL = detect_label_col(sample_df)
feature_cols = [c for c in sample_df.columns if c != LABEL_COL and pd.api.types.is_numeric_dtype(sample_df[c])]

print("\nLabel column:", LABEL_COL)
print("Numeric feature count:", len(feature_cols))
print("Example features:", feature_cols[:8])

# Class balance
train_counts = stream_label_counts(train_parts, feature_cols, LABEL_COL)
val_counts = stream_label_counts(val_parts, feature_cols, LABEL_COL)
test_counts = stream_label_counts(test_parts, feature_cols, LABEL_COL)

print("\nClass balance (valid labels only):")
print("TRAIN:", train_counts, proportions(train_counts))
print("VAL  :", val_counts, proportions(val_counts))
print("TEST :", test_counts, proportions(test_counts))

labels = ["TRAIN", "VAL", "TEST"]
benign = [train_counts[0], val_counts[0], test_counts[0]]
ddos = [train_counts[1], val_counts[1], test_counts[1]]

plt.figure()
x = np.arange(len(labels))
plt.bar(x - 0.2, benign, width=0.4, label="Benign (0)")
plt.bar(x + 0.2, ddos, width=0.4, label="DDoS (1)")
plt.xticks(x, labels)
plt.yscale("log")
plt.title("Class Balance by Split (log scale)")
plt.xlabel("Split")
plt.ylabel("Count (log)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_class_balance_by_split.png", dpi=200)
plt.close()
print("Saved: fig_class_balance_by_split.png")

# Dummy baseline
dummy = DummyClassifier(strategy="most_frequent")
fitted = False
for p in train_parts:
    X, y = read_xy_part(p, feature_cols, LABEL_COL)
    if y.size == 0:
        continue
    dummy.fit(X, y)
    fitted = True
    break
if not fitted:
    raise ValueError("Could not fit dummy: no valid labels found in TRAIN after filtering.")

cm_val_dummy, acc_val_dummy, f1_val_dummy, _, _, _ = eval_stream(
    val_parts, feature_cols, LABEL_COL, predict_fn=dummy.predict, need_scores=False
)
cm_test_dummy, acc_test_dummy, f1_test_dummy, _, _, _ = eval_stream(
    test_parts, feature_cols, LABEL_COL, predict_fn=dummy.predict, need_scores=False
)

print("\nDummy baseline:")
print("VAL  accuracy:", acc_val_dummy, "F1 macro:", f1_val_dummy)
print("TEST accuracy:", acc_test_dummy, "F1 macro:", f1_test_dummy)

save_cm(cm_val_dummy, "Confusion Matrix (Dummy) - Validation", FIG_DIR / "fig_cm_dummy_val.png")
save_cm(cm_test_dummy, "Confusion Matrix (Dummy) - Friday Test", FIG_DIR / "fig_cm_dummy_test.png")
print("Saved: fig_cm_dummy_val.png")
print("Saved: fig_cm_dummy_test.png")

# SGD logistic regression (streamed scaler + partial_fit)
scaler = StandardScaler(with_mean=False)
clf_sgd = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    max_iter=1,
    tol=None,
    random_state=RANDOM_STATE
)
classes = np.array([0, 1], dtype=np.int64)

print("\nTraining SGD logistic regression (streaming)...")

for i, p in enumerate(train_parts, 1):
    X, y = read_xy_part(p, feature_cols, LABEL_COL)
    if y.size == 0:
        continue
    for start in range(0, X.shape[0], CHUNK_SIZE):
        scaler.partial_fit(X[start:start + CHUNK_SIZE])
    if i % 10 == 0 or i == len(train_parts):
        print(f"Scaler processed {i}/{len(train_parts)} train parts")

trained = False
seen = 0
for i, p in enumerate(train_parts, 1):
    X, y = read_xy_part(p, feature_cols, LABEL_COL)
    if y.size == 0:
        continue
    for start in range(0, X.shape[0], CHUNK_SIZE):
        Xb = scaler.transform(X[start:start + CHUNK_SIZE])
        yb = y[start:start + CHUNK_SIZE]
        if not trained:
            clf_sgd.partial_fit(Xb, yb, classes=classes)
            trained = True
        else:
            clf_sgd.partial_fit(Xb, yb)
        seen += len(yb)
    if i % 10 == 0 or i == len(train_parts):
        print(f"SGD | part {i}/{len(train_parts)} | seen={seen}")

if not trained:
    raise ValueError("SGD was not trained: no valid labelled data found in TRAIN after filtering.")

dump({"scaler": scaler, "model": clf_sgd}, MODELS_DIR / "sgd_logreg_stream.joblib")
print("Saved: sgd_logreg_stream.joblib")

cm_val_sgd, acc_val_sgd, f1_val_sgd, auc_val_sgd, _, _ = eval_stream(
    val_parts,
    feature_cols,
    LABEL_COL,
    predict_fn=clf_sgd.predict,
    score_fn=lambda X: clf_sgd.predict_proba(X)[:, 1],
    scaler=scaler,
    need_scores=True
)
cm_test_sgd, acc_test_sgd, f1_test_sgd, auc_test_sgd, y_test_sgd, yscore_test_sgd = eval_stream(
    test_parts,
    feature_cols,
    LABEL_COL,
    predict_fn=clf_sgd.predict,
    score_fn=lambda X: clf_sgd.predict_proba(X)[:, 1],
    scaler=scaler,
    need_scores=True
)

print("\nSGD-LogReg:")
print("VAL  accuracy:", acc_val_sgd, "F1 macro:", f1_val_sgd, "ROC AUC:", auc_val_sgd)
print("TEST accuracy:", acc_test_sgd, "F1 macro:", f1_test_sgd, "ROC AUC:", auc_test_sgd)

save_cm(cm_val_sgd, "Confusion Matrix (SGD) - Validation", FIG_DIR / "fig_cm_sgd_val.png")
save_cm(cm_test_sgd, "Confusion Matrix (SGD) - Friday Test", FIG_DIR / "fig_cm_sgd_test.png")
_ = save_roc(y_test_sgd, yscore_test_sgd, "ROC (SGD) - Friday Test", FIG_DIR / "fig_roc_sgd_test.png")

print("Saved: fig_cm_sgd_val.png")
print("Saved: fig_cm_sgd_test.png")
print("Saved: fig_roc_sgd_test.png")

coef = clf_sgd.coef_.ravel()
top_k = 20
idx = np.argsort(np.abs(coef))[::-1][:top_k]
plt.figure(figsize=(10, 6))
plt.barh(np.array(feature_cols, dtype=object)[idx][::-1], coef[idx][::-1])
plt.title("Top SGD Coefficients by Absolute Magnitude")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_coefficients_sgd_logreg.png", dpi=200)
plt.close()
print("Saved: fig_coefficients_sgd_logreg.png")

# Random forest on stratified sample
TARGET_N = 200_000
ratio_pos = train_counts[1] / max(1, (train_counts[0] + train_counts[1]))
target_pos = int(round(TARGET_N * ratio_pos))
target_neg = TARGET_N - target_pos

rng = np.random.default_rng(RANDOM_STATE)
got = {0: 0, 1: 0}
xs = []
ys = []

print("\nSampling stratified subset for RF...")
for p in train_parts:
    if got[0] >= target_neg and got[1] >= target_pos:
        break

    X, y = read_xy_part(p, feature_cols, LABEL_COL)
    if y.size == 0:
        continue

    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]

    need0 = max(0, target_neg - got[0])
    need1 = max(0, target_pos - got[1])

    if need0 > 0 and idx0.size > 0:
        take0 = min(need0, idx0.size)
        pick0 = rng.choice(idx0, size=take0, replace=False)
        xs.append(X[pick0])
        ys.append(y[pick0])
        got[0] += take0

    if need1 > 0 and idx1.size > 0:
        take1 = min(need1, idx1.size)
        pick1 = rng.choice(idx1, size=take1, replace=False)
        xs.append(X[pick1])
        ys.append(y[pick1])
        got[1] += take1

    print(f"Sampling progress: got={got} from {p.name}")

X_rf = np.vstack(xs).astype(np.float32, copy=False)
y_rf = np.concatenate(ys).astype(np.int64, copy=False)

print("RF sample:", X_rf.shape, "balance:", {0: float(np.mean(y_rf == 0)), 1: float(np.mean(y_rf == 1))})

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("Training RF on stratified sample...")
rf.fit(X_rf, y_rf)

dump(rf, MODELS_DIR / "rf_sample.joblib")
print("Saved: rf_sample.joblib")

cm_val_rf, acc_val_rf, f1_val_rf, auc_val_rf, _, _ = eval_stream(
    val_parts,
    feature_cols,
    LABEL_COL,
    predict_fn=rf.predict,
    score_fn=lambda X: rf.predict_proba(X)[:, 1],
    scaler=None,
    need_scores=True
)
cm_test_rf, acc_test_rf, f1_test_rf, auc_test_rf, y_test_rf, yscore_test_rf = eval_stream(
    test_parts,
    feature_cols,
    LABEL_COL,
    predict_fn=rf.predict,
    score_fn=lambda X: rf.predict_proba(X)[:, 1],
    scaler=None,
    need_scores=True
)

print("\nRandom Forest:")
print("VAL  accuracy:", acc_val_rf, "F1 macro:", f1_val_rf, "ROC AUC:", auc_val_rf)
print("TEST accuracy:", acc_test_rf, "F1 macro:", f1_test_rf, "ROC AUC:", auc_test_rf)

save_cm(cm_val_rf, "Confusion Matrix (RF) - Validation", FIG_DIR / "fig_cm_rf_val.png")
save_cm(cm_test_rf, "Confusion Matrix (RF) - Friday Test", FIG_DIR / "fig_cm_rf_test.png")
_ = save_roc(y_test_rf, yscore_test_rf, "ROC (RF) - Friday Test", FIG_DIR / "fig_roc_rf_test.png")

print("Saved: fig_cm_rf_val.png")
print("Saved: fig_cm_rf_test.png")
print("Saved: fig_roc_rf_test.png")

importances = rf.feature_importances_
idx_imp = np.argsort(importances)[::-1]
top_k = 20
top_features = [(feature_cols[i], float(importances[i])) for i in idx_imp[:top_k]]

plt.figure(figsize=(10, 6))
plt.barh([f for f, _ in top_features][::-1], [v for _, v in top_features][::-1])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_feature_importance_rf.png", dpi=200)
plt.close()
print("Saved: fig_feature_importance_rf.png")

top_feat_path = PREP_DIR / "rf_top_features.csv"
pd.DataFrame(top_features, columns=["feature", "importance"]).to_csv(top_feat_path, index=False)
print("Saved:", str(top_feat_path))

metrics_rows = [
    {"model": "Dummy baseline", "split": "Validation", "accuracy": acc_val_dummy, "f1_macro": f1_val_dummy, "roc_auc": ""},
    {"model": "Dummy baseline", "split": "FridayTest", "accuracy": acc_test_dummy, "f1_macro": f1_test_dummy, "roc_auc": ""},
    {"model": "SGD Logistic Regression", "split": "Validation", "accuracy": acc_val_sgd, "f1_macro": f1_val_sgd, "roc_auc": auc_val_sgd},
    {"model": "SGD Logistic Regression", "split": "FridayTest", "accuracy": acc_test_sgd, "f1_macro": f1_test_sgd, "roc_auc": auc_test_sgd},
    {"model": "Random Forest (sample)", "split": "Validation", "accuracy": acc_val_rf, "f1_macro": f1_val_rf, "roc_auc": auc_val_rf},
    {"model": "Random Forest (sample)", "split": "FridayTest", "accuracy": acc_test_rf, "f1_macro": f1_test_rf, "roc_auc": auc_test_rf},
]
metrics_out = MET_DIR / "metrics_summary.csv"
pd.DataFrame(metrics_rows).to_csv(metrics_out, index=False)
print("\nSaved:", str(metrics_out))
print("\nDONE.")


Project root: C:\Users\yemio\Downloads\DSAC CW1\DSAC
Processed parts dir: C:\Users\yemio\Downloads\DSAC CW1\DSAC\data\processed

Label column: Label
Numeric feature count: 80
Example features: ['Source Port', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets']

Class balance (valid labels only):
TRAIN: {0: 1485909, 1: 214241} {0: 0.8739870011469576, 1: 0.12601299885304237}
VAL  : {0: 371478, 1: 53530} {0: 0.8740494296577946, 1: 0.12595057034220533}
TEST : {0: 26135, 1: 23865} {0: 0.5227, 1: 0.4773}
Saved: fig_class_balance_by_split.png

Dummy baseline:
VAL  accuracy: 0.8740494296577946 F1 macro: 0.4663961450672077
TEST accuracy: 0.5227 F1 macro: 0.3432718197937874
Saved: fig_cm_dummy_val.png
Saved: fig_cm_dummy_test.png

Training SGD logistic regression (streaming)...
Scaler processed 10/39 train parts
Scaler processed 20/39 train parts
Scaler processed 30/39 train parts
Scaler pro

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

print("Running feature robustness experiment")
print("Processed dir:", PROCESSED_DIR)

CHUNK_SIZE = 50_000
RANDOM_STATE = 42


def list_parts(prefix: str) -> list[Path]:
    parts = sorted(PROCESSED_DIR.glob(f"{prefix}_part_*.parquet"))
    if not parts:
        raise FileNotFoundError(f"No Parquet parts found for prefix='{prefix}' in {PROCESSED_DIR}")
    return parts


def coerce_binary_labels(series: pd.Series) -> pd.Series:
    s = series.copy()
    if s.dtype == object:
        ss = s.astype(str).str.strip()
        mapped = ss.map({"BENIGN": 0, "DDoS": 1})
        mapped = mapped.where(~mapped.isna(), (ss != "BENIGN").astype(int))
        return pd.to_numeric(mapped, errors="coerce")
    return pd.to_numeric(s, errors="coerce")


def read_xy_part(p: Path, feature_cols: list[str], label_col: str) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_parquet(p, columns=feature_cols + [label_col])
    y = coerce_binary_labels(df[label_col])
    valid = y.isin([0, 1]).to_numpy()
    if valid.sum() == 0:
        return np.empty((0, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.int64)
    X = df.loc[valid, feature_cols].to_numpy(dtype=np.float32, copy=False)
    yv = y.loc[valid].to_numpy(dtype=np.int64, copy=False)
    return X, yv


def save_cm(cm: np.ndarray, title: str, out_path: Path) -> None:
    plt.figure()
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(int(v)), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


def save_roc(y_true: np.ndarray, y_score: np.ndarray, title: str, out_path: Path) -> float:
    auc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(f"{title} (AUC={auc:.3f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return auc


def eval_stream(parts: list[Path], feature_cols: list[str], label_col: str, scaler, clf):
    cm = np.zeros((2, 2), dtype=np.int64)
    y_true_all = []
    y_pred_all = []
    y_score_all = []

    for p in parts:
        X, y = read_xy_part(p, feature_cols, label_col)
        if y.size == 0:
            continue

        Xs = scaler.transform(X)
        y_pred = clf.predict(Xs)
        y_score = clf.predict_proba(Xs)[:, 1]

        cm += confusion_matrix(y, y_pred, labels=[0, 1])

        y_true_all.append(y)
        y_pred_all.append(y_pred)
        y_score_all.append(y_score)

    if not y_true_all:
        raise ValueError("No valid labels found in this split after coercion/filtering.")

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)
    y_score = np.concatenate(y_score_all)

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    auc = roc_auc_score(y_true, y_score)

    return cm, acc, f1m, auc, y_true, y_score


robust_figs = [
    "fig_cm_sgd_robust_val.png",
    "fig_cm_sgd_robust_test.png",
    "fig_roc_sgd_robust_test.png",
]
for fn in robust_figs:
    fp = FIG_DIR / fn
    if fp.exists():
        fp.unlink()

robust_csv = MET_DIR / "metrics_summary_robust.csv"
if robust_csv.exists():
    robust_csv.unlink()

# Parts + schema
train_parts = list_parts("train_clean")
val_parts = list_parts("val_clean")
test_parts = list_parts("test_clean")

sample_df = pd.read_parquet(train_parts[0])
label_col = LABEL_COL
all_features = [c for c in sample_df.columns if c != label_col and pd.api.types.is_numeric_dtype(sample_df[c])]

top_feat_path = PREP_DIR / "rf_top_features.csv"
if not top_feat_path.exists():
    raise FileNotFoundError(f"Missing {top_feat_path}. Run the main pipeline cell first.")

top_df = pd.read_csv(top_feat_path)
drop_features = top_df["feature"].head(5).tolist()

print("Dropping features:", drop_features)

feature_cols = [c for c in all_features if c not in drop_features]
print("Robust feature count:", len(feature_cols))

# Streamed scaler + streamed SGD
scaler = StandardScaler(with_mean=False)
clf = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    learning_rate="optimal",
    max_iter=1,
    tol=None,
    random_state=RANDOM_STATE
)

for i, p in enumerate(train_parts, 1):
    X, y = read_xy_part(p, feature_cols, label_col)
    if y.size == 0:
        continue
    for start in range(0, X.shape[0], CHUNK_SIZE):
        scaler.partial_fit(X[start:start + CHUNK_SIZE])
    if i % 10 == 0 or i == len(train_parts):
        print(f"Scaler fit on {i}/{len(train_parts)} train parts")

classes = np.array([0, 1], dtype=np.int64)
trained = False
for i, p in enumerate(train_parts, 1):
    X, y = read_xy_part(p, feature_cols, label_col)
    if y.size == 0:
        continue
    for start in range(0, X.shape[0], CHUNK_SIZE):
        Xb = scaler.transform(X[start:start + CHUNK_SIZE])
        yb = y[start:start + CHUNK_SIZE]
        if not trained:
            clf.partial_fit(Xb, yb, classes=classes)
            trained = True
        else:
            clf.partial_fit(Xb, yb)
    if i % 10 == 0 or i == len(train_parts):
        print(f"Trained on {i}/{len(train_parts)} train parts")

if not trained:
    raise ValueError("Robust SGD was not trained: no valid labelled data found in TRAIN after filtering.")

print("Robust SGD trained.")

cm_val, acc_val, f1_val, auc_val, _, _ = eval_stream(val_parts, feature_cols, label_col, scaler, clf)
cm_test, acc_test, f1_test, auc_test, y_test, yscore_test = eval_stream(test_parts, feature_cols, label_col, scaler, clf)

print("\nSGD-Robust — VALIDATION")
print("Accuracy:", acc_val)
print("F1 macro:", f1_val)
print("ROC AUC:", auc_val)

print("\nSGD-Robust — TEST (Friday)")
print("Accuracy:", acc_test)
print("F1 macro:", f1_test)
print("ROC AUC:", auc_test)

save_cm(cm_val, "Confusion Matrix (SGD-Robust) - Validation", FIG_DIR / "fig_cm_sgd_robust_val.png")
save_cm(cm_test, "Confusion Matrix (SGD-Robust) - Friday Test", FIG_DIR / "fig_cm_sgd_robust_test.png")
_ = save_roc(y_test, yscore_test, "ROC (SGD-Robust) - Friday Test", FIG_DIR / "fig_roc_sgd_robust_test.png")

print("Saved robust figures:",
      "fig_cm_sgd_robust_val.png, fig_cm_sgd_robust_test.png, fig_roc_sgd_robust_test.png")

pd.DataFrame([
    {"model": "SGD Robust (drop-5)", "split": "Validation", "accuracy": acc_val, "f1_macro": f1_val, "roc_auc": auc_val},
    {"model": "SGD Robust (drop-5)", "split": "FridayTest", "accuracy": acc_test, "f1_macro": f1_test, "roc_auc": auc_test},
]).to_csv(MET_DIR / "metrics_summary_robust.csv", index=False)

print("Saved:", str(MET_DIR / "metrics_summary_robust.csv"))


Running feature robustness experiment
Processed dir: C:\Users\yemio\Downloads\DSAC CW1\DSAC\data\processed
Dropping features: ['Bwd Packet Length Std', 'Packet Length Std', 'Destination Port', 'Bwd Packet Length Max', 'Max Packet Length']
Robust feature count: 75
Scaler fit on 10/39 train parts
Scaler fit on 20/39 train parts
Scaler fit on 30/39 train parts
Scaler fit on 39/39 train parts
Trained on 10/39 train parts
Trained on 20/39 train parts
Trained on 30/39 train parts
Trained on 39/39 train parts
Robust SGD trained.

SGD-Robust — VALIDATION
Accuracy: 0.9621583593720588
F1 macro: 0.9041305368444268
ROC AUC: 0.9806981908501484

SGD-Robust — TEST (Friday)
Accuracy: 0.83856
F1 macro: 0.8326423955004232
ROC AUC: 0.9778187993965644
Saved robust figures: fig_cm_sgd_robust_val.png, fig_cm_sgd_robust_test.png, fig_roc_sgd_robust_test.png
Saved: C:\Users\yemio\Downloads\DSAC CW1\DSAC\report\metrics\metrics_summary_robust.csv
